# Assignment — Bloc 4a, Sessió 6
### Mini-repte: Analitzar i crear fitxers MIDI amb `mido`

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Per guardar els teus canvis: `File → Save a copy in Drive`.

**Objectiu:** Llegir i analitzar un fitxer MIDI existent, i crear-ne un de propi. Completa les funcions marcades amb `# TODO`. Cada part té una cel·la d'autotest.

No esborris les cel·les d'autotest.

In [ ]:
!apt-get install -y fluidsynth fluid-soundfont-gm -q > /dev/null
!pip install mido pretty_midi pyfluidsynth -q
import mido
import pretty_midi
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import urllib.request

base_url = "https://raw.githubusercontent.com/jcomajuncosas/tp2/main/04_midi/sessio06_estructura_midi/recursos/midi/"
urllib.request.urlretrieve(base_url + "example_scale.mid", "example_scale.mid")
print("Fitxers carregats")

## Part 1 — Analitzar un fitxer MIDI

Implementa `compta_notes`: rep un track de `mido` i retorna el nombre total de notes (missatges `note_on` amb velocity > 0).

In [ ]:
def compta_notes(track):
    """Retorna el nombre de note_on (velocity > 0) en un track."""
    # TODO 1: compta els missatges de tipus 'note_on' amb velocity > 0
    # Pista: itera 'track', comprova msg.type i msg.velocity
    return None  # <-- substitueix

In [ ]:
# AUTOTEST 1
mid = mido.MidiFile('example_scale.mid')
n_notes_melodia = compta_notes(mid.tracks[1])
n_notes_baix    = compta_notes(mid.tracks[2])

assert n_notes_melodia is not None, "La funció retorna None"
assert n_notes_melodia == 16, \
    f"La melodia hauria de tenir 16 notes (escala amunt i avall), té {n_notes_melodia}"
assert n_notes_baix == 8, \
    f"El baix hauria de tenir 8 notes, té {n_notes_baix}"

print(f"✅ Part 1 correcta! Melodia: {n_notes_melodia} notes, Baix: {n_notes_baix} notes")

## Part 2 — Extreure la nota més alta i la més baixa

Implementa `rang_notes`: rep un track i retorna `(nota_minima, nota_maxima)`.

In [ ]:
def rang_notes(track):
    """Retorna (nota_minima, nota_maxima) del track."""
    # TODO 2: troba la nota MIDI mínima i màxima entre tots els note_on (velocity>0)
    # Pista: pots usar una llista i min()/max(), o actualitzar variables
    return None, None  # <-- substitueix

In [ ]:
# AUTOTEST 2
noms = ['Do','Do#','Re','Re#','Mi','Fa','Fa#','Sol','Sol#','La','La#','Si']

nota_min, nota_max = rang_notes(mid.tracks[1])

assert nota_min is not None, "La funció retorna None"
assert nota_min == 60, f"Nota mínima hauria de ser 60 (Do4), és {nota_min}"
assert nota_max == 72, f"Nota màxima hauria de ser 72 (Do5), és {nota_max}"

print(f"✅ Part 2 correcta!")
print(f"   Rang melodia: {nota_min} ({noms[nota_min%12]}{nota_min//12-1}) — "
      f"{nota_max} ({noms[nota_max%12]}{nota_max//12-1})")

## Part 3 — Crear una seqüència MIDI pròpia

Crea un fitxer MIDI amb **almenys 8 notes** seguint un patró musical de la teva elecció (escala, acord arpegiado, melodia curta...). Ha d'incloure:
- Track de metadades amb tempo
- Track de notes amb `note_on` i `note_off` correctes (delta time ben posat)
- Variació de velocity entre notes (no totes iguals)

**Connexió amb TP1:** és exactament el que feies amb `for note in pattern: piano(note, vel, dur)`, però ara generes un fitxer `.mid` persistent.

In [ ]:
# TODO 3: crea el teu fitxer MIDI

mid_meu = mido.MidiFile(ticks_per_beat=480)
tempo = mido.bpm2tempo(120)
quarter = 480
eighth  = 240

# Track de metadades
meta = mido.MidiTrack()
meta.append(mido.MetaMessage('set_tempo', tempo=tempo, time=0))
mid_meu.tracks.append(meta)

# TODO 3a: defineix el teu patró de notes (almenys 8)
pattern = []  # <-- substitueix amb les teves notes MIDI

# TODO 3b: defineix les velocitats (una per nota, valors entre 40 i 120)
velocitats = []  # <-- substitueix (mateixa longitud que pattern)

# TODO 3c: defineix les durades en ticks (una per nota)
# quarter=480 (negra), eighth=240 (corxera), quarter*2=960 (blanca)
durades = []  # <-- substitueix (mateixa longitud que pattern)

# TODO 3d: crea el track i afegeix les notes
track = mido.MidiTrack()
# per a cada nota: note_on (time=0) + note_off (time=durada)

track.append(mido.MetaMessage('end_of_track', time=0))
mid_meu.tracks.append(track)
mid_meu.save('la_meva_sequencia.mid')
print("Fitxer creat: la_meva_sequencia.mid")

In [ ]:
# AUTOTEST 3
mid_check = mido.MidiFile('la_meva_sequencia.mid')

assert len(mid_check.tracks) >= 2, "Ha d'haver almenys 2 tracks (metadades + notes)"

n = compta_notes(mid_check.tracks[1])
assert n is not None and n >= 8, \
    f"Ha d'haver almenys 8 notes, n'hi ha {n}"

nota_min_check, nota_max_check = rang_notes(mid_check.tracks[1])
assert nota_min_check is not None

# Comprova variació de velocity
velocitats_trobades = set()
for msg in mid_check.tracks[1]:
    if msg.type == 'note_on' and msg.velocity > 0:
        velocitats_trobades.add(msg.velocity)
assert len(velocitats_trobades) > 1, \
    "Les velocitats han de variar (almenys 2 valors diferents)"

# Comprova delta time: cap note_on ha de tenir time > 0 (seria un silenci involuntari)
for msg in mid_check.tracks[1]:
    if msg.type == 'note_on' and msg.velocity > 0:
        assert msg.time == 0, \
            f"note_on amb time={msg.time} — el time ha de ser 0 al note_on (el retard va al note_off)"

print(f"✅ Part 3 correcta!")
print(f"   Notes: {n}, Rang: {nota_min_check}-{nota_max_check}, "
      f"Velocitats diferents: {len(velocitats_trobades)}")

# Escoltem el resultat
pm = pretty_midi.PrettyMIDI('la_meva_sequencia.mid')
audio = pm.fluidsynth(fs=44100)
display(Audio(audio, rate=44100))

## Part 4 — Visualitza el piano roll de la teva seqüència

In [ ]:
# TODO 4: visualitza el piano roll de 'la_meva_sequencia.mid'
# Pots reutilitzar el codi de visualització dels Patches
# (la funció extrau_notes + barh)

# El teu codi aquí

---
## 🚀 Challenges (opcional)

1. **Dues veus:** afegeix un segon track al teu fitxer (baix, acompanyament o contrapunt).
2. **Tempo variable:** afegeix missatges `set_tempo` al llarg del track per crear un accelerando o ritardando.
3. **Converter velocity:** escriu una funció que llegeixi un fitxer MIDI i retorni un histograma de velocitats (quantes notes hi ha per cada rang de velocity: pp, p, mf, f, ff).
4. **Transposició:** escriu una funció `transposa(fitxer_entrada, semitots, fitxer_sortida)` que llegeixi un `.mid`, transposi totes les notes N semitots, i desi el resultat.

In [ ]:
# El teu codi del Challenge aquí